In [1]:
!wget -q -O - https://dl-ssl.google.com/linux/linux_signing_key.pub | apt-key add -
!echo 'deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main' | tee /etc/apt/sources.list.d/google-chrome.list
!apt-get update
!apt-get install google-chrome-stable
!pip install selenium
!pip install webdriver-manager
from datetime import datetime
import re
import time

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
def extract_stock_data(html):
    '''
    株価データを抽出する関数

    Parameters
    ----------
    html : str
        ページソース

    Returns
    -------
    list
        日付と株価データのリスト
    '''

    soup = BeautifulSoup(html, 'html.parser')
    graph = soup.find('div', class_='highcharts-tooltip')

    graph_td = graph.find_all('td')

    # 日付を変換してリストの先頭に追加
    date = datetime.strptime(graph_td[0].text, '%Y/%m/%d').date().strftime('%Y-%m-%d')
    values = [date]
    for value in graph_td[1:]:
        # 始値、高値、安値、終値を抽出してリストに追加
        if re.findall('始値|高値|安値|終値', value.text):
            values.append(re.sub(r'\D', '', value.text))

    return values


def get_stock_values(driver, url):
    '''
    株価データを取得する関数

    Parameters
    ----------
    driver : selenium.webdriver
        Selenium WebDriverオブジェクト
    url : str
        株価データを取得するWebページのURL

    Returns
    -------
    list
        日付と株価データのリストのリスト
    '''

    stock_values = []

    # 指定されたURLにアクセスする
    driver.get(url)

    # グラフ要素を最大で10回の試行で取得する
    for _ in range(10):
        graph_xy = driver.find_elements(By.CSS_SELECTOR, '.highcharts-grid')[1]
        if graph_xy is not None:
            break
        print('continue find elements')

    # グラフ要素の幅を取得する
    graph_width = graph_xy.rect['width']

    # グラフの中央にマウスポインタを移動する
    actions = ActionChains(driver)
    actions.move_to_element(graph_xy).perform()
    actions.move_by_offset(graph_width // 2, 0).perform()

    # ページソースから株価データを抽出する
    html = driver.page_source.encode('utf-8')
    stock_values.append(extract_stock_data(html))

    # グラフを1ピクセル左に移動しながら、株価データを抽出する
    for _ in range(graph_width - 1):
        actions = ActionChains(driver)
        actions.move_by_offset(-1, 0).perform()
        html = driver.page_source.encode('utf-8')
        temp_value = extract_stock_data(html)
        if temp_value not in stock_values:
            stock_values.append(temp_value)

    return stock_values

chart_type = '6month'
url = f'https://www.nikkei.com/markets/worldidx/chart/nk225/?type={chart_type}'

# ヘッドレスモードで起動するためのオプションを設定
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

# ChromeDriverManagerを使用してChromeDriverを自動的に管理
service = Service(ChromeDriverManager().install())

# Google Chrome用のブラウザドライバーインスタンスを作成
chrome_driver = webdriver.Chrome(service=service, options=chrome_options)
start_time = time.time()

# 株価データを取得する
result = get_stock_values(chrome_driver, url)
print(f'Scraping time: {time.time() - start_time}')

# ブラウザを閉じる
chrome_driver.quit()

# 結果を表示する
print('日付, 始値, 高値, 安値, 終値')
for data in result:
    print(', '.join(data))

OK
deb [arch=amd64] http://dl.google.com/linux/chrome/deb/ stable main
Get:1 http://dl.google.com/linux/chrome/deb stable InRelease [1,825 B]
Hit:2 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:3 https://cli.github.com/packages stable InRelease [3,917 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:9 http://dl.google.com/linux/chrome/deb stable/main amd64 Packages [1,213 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Hit:12 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:13 https://cloud.r-project.org/bin/linux